# Capstone --- Chapter 9: Memory as a Trained Triple Store

Chapter~9 distinguishes three access patterns an agent's memory must serve --- recency, similarity and relationships --- and adds a fourth tier for correctness: a trained geometric triple store that recalls exact values and rejects contradictions on write. This companion reads that fourth tier on the capstone banking complaint agent, whose policy knowledge is precisely such a store. A customer question is not matched to a passage by vector similarity; it is parsed into a (head, relation, tail) query triple, bound to the graph's canonical vocabulary and answered by the governing fact the graph actually holds.

## The problem with similarity retrieval for a governing fact

Consider the question *"How much is the overdraft fee?"*. A vector store returns the passage whose embedding lies nearest the question, and synthesis reads a number out of that prose. Two failure modes follow. First, an adjacent-but-wrong attribute --- the overdraft *interest rate* rather than the *fee* --- embeds close enough to be retrieved and paraphrased as the answer. Second, the number itself is decoded by a language model from prose, so it can drift. The capstone's retriever instead treats the relation as a geometric operator: the question binds to the policy *entity* (`overdraft`), and the fee is read as the exact tail of the `overdraft --has_fee_amount--> ?` edge the graph asserts. Memory here is a structured store queried by structure, not a passage index queried by proximity.

## The retriever and its store

`PolicyRagRetriever` in `agentlab.capstone.policy_rag` drives knowlytix's GEODE graph-RAG pipeline over a self-corrected geometric store built from the banking policy corpus. Construction loads the trained store, the document-tuned encoder and the local query-time model, so it is deferred to the reader. The cell below confirms only that the symbols import.

In [ ]:
from agentlab.capstone import policy_rag

# The default store the deployed retriever loads (a trained triple store,
# not a vector index). Construction is GPU/heavy; we only report the path.
print('default store :', policy_rag._DEFAULT_STORE)
print('query-time LLM :', policy_rag._RAG_LLM_MODEL)

## Parse, then bind: from a customer message to a query triple

The parse step is isolated on the retriever as `extract`. It runs the GEODE parse-and-bind loop on a raw message and returns the grounded extraction: the query triples the question was parsed into, each bound to the graph's real vocabulary so colloquial wording (*"I overdrew my account"*) maps onto the canonical policy entity (`overdraft`). The return carries `query_facts` --- the bound `(head, relation, tail)` triples --- and `is_bound`, which is `False` when nothing grounds, the abstain that keeps a vague message from acquiring a fabricated label.

This cell issues a live parse and requires the loaded store, so it is written for the reader to run; it is not executed in this notebook.

In [ ]:
# READER-RUNNABLE (loads the GPU store + encoder). Not executed here.
#
# retriever = get_default_retriever()
# parsed = retriever.extract('I was charged a $35 overdraft fee I did not authorize.')
# print('is_bound    :', parsed['is_bound'])
# for h, r, t in parsed['query_facts']:
#     print(f'  {h} --{r}--> {t}')
#
# Expected shape: is_bound=True and a bound triple headed by the policy
# entity, e.g.  overdraft --has_fee_amount--> ?  (the tail is the slot the
# retrieval fills from the graph).

## Retrieve the governing fact

`search` routes the query through the pipeline and returns the grounded answer in the `search_policy` tool's shape. The bound head's admissible facts are retrieved through the relation operators, an answering fact is selected (or the pipeline abstains), and the result carries the resolved policy `id`, the retrieved `policies` ranked by plausibility, the provenance `text`, the synthesized `answer`, a confidence `score` and the `query_facts` that bound. An `extraction` from a prior `extract` call may be passed back so the message is parsed once. This cell also requires the loaded store and is left for the reader to run.

In [ ]:
# READER-RUNNABLE (loads the GPU store + query-time LLM). Not executed here.
#
# retriever = get_default_retriever()
# hits = retriever.search('How much is the overdraft fee?', k=3)
# top = hits[0]
# print('policy id   :', top['id'])
# print('policies    :', top['policies'])       # plausibility-ranked recall
# print('answer      :', top['answer'])
# print('score       :', top['score'], 'decision:', top['decision'])
# print('query_facts :', top['query_facts'])    # the bound (h, r, t) triples
# print('provenance  :')
# print(top['text'])
#
# An empty list is the honest abstain: nothing bound, or the answer rested
# on inadmissible evidence (the cap/tension gate dropped every retrieved fact).

## What the structured store recovers, at graph truth

To read the effect of structured retrieval without loading the store, we inspect a pinned artifact of graph-truth numbers over a fixed question set. It reports the triple store (`gms`) against a similarity baseline (`dense`) on the same questions. The store binds every question (`parse_rate` and `bind_rate` at 1.0) and recovers the governing fact at markedly higher precision than top-k similarity --- the direct measurement of the failure mode the opening section described.

In [ ]:
import json
from pathlib import Path

root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code')) if (c / 'data').exists()), Path('.'))
art = json.loads((root / 'data' / 'capstone_retrieval.json').read_text())
gms, dense = art['gms'], art['dense']
print(f"questions        : {art['n']}")
print(f"gms   recall     : {gms['recall']:.2f}   precision  : {gms['precision']:.2f}")
print(f"gms   parse_rate : {gms['parse_rate']:.2f}   bind_rate  : {gms['bind_rate']:.2f}")
print(f"dense recall     : {dense['recall']:.2f}   prec@1     : {dense['precision_at_1']:.2f}   prec@k : {dense['precision_at_k']:.2f}")

## A miss is a bound triple, not a lost passage

Where the store misses, the artifact records *which* query triple was constructed and what the graph returned for it. A miss is legible: the question parsed and bound to a `(head, relation, tail)`, and the retrieved tail was the wrong register (a per-occurrence question answered by a fee amount). This is the diagnostic value of memory-as-structure --- a failure names the triple it resolved, rather than a passage that happened to rank.

In [ ]:
for m in art.get('misses', []):
    (h, r, t), = m['triples']
    got = m['answers'][0][0] if m.get('answers') else '(none)'
    print(f"Q: {m['question']}")
    print(f"   triple   : {h} --{r}--> {t}")
    print(f"   expected : {m['expected']:<16} got: {got}   (missed at: {m['stage']})")
    print()

This is the capstone's realization of Chapter~9. The policy knowledge that `search_policy` draws on is the chapter's fourth memory tier: a trained geometric triple store, queried by parse-and-bind rather than by vector similarity, so a governing fact is recovered as the exact tail of the relation the question names and a miss is a legible bound triple. The three general access patterns of Chapter~9 --- recency, similarity, relationships --- remain the agent's working memory; correctness-critical policy lookup rests on the structured tier. Chapter~16 assembles this retriever into the governed complaint workflow, where the retrieved fact and its provenance span constrain the drafted response.